# Exact final-set accuracy from 100 repeats

This notebook reads 100-repeat iteration outputs, resolves each repeat's `final_set_of_gene` back to Ensembl IDs, and checks accuracy in two ways:

- per repeat: whether the final set contains any causal gene
- per disease after 100 repeats: whether the union of all final sets contains the whole causal-gene set

In [13]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/miasmacbook/Desktop/kcl/6-months_project")

# Change this label and root when checking another module.
# Examples:
#   MODULE_LABEL = "base";        ITERATION_OUTPUT_ROOTS = [PROJECT_ROOT / "pocket_model_iterations"]
#   MODULE_LABEL = "left1_right1"; ITERATION_OUTPUT_ROOTS = [PROJECT_ROOT / "pocket_model_iterations_2_in_total"]
#   MODULE_LABEL = "left2_right2"; ITERATION_OUTPUT_ROOTS = [PROJECT_ROOT / "pocket_model_iterations_4_in_total"]
MODULE_LABEL = "4"
ITERATION_OUTPUT_ROOTS = [
    PROJECT_ROOT / "pocket_model_iterations_4_in_total",
]

# Causal-gene and pocket-candidate files are expected at:
#   DISEASE_DATA_ROOT/<n>_gene_related_disease/<disease_id>/<disease_id>.csv
#   DISEASE_DATA_ROOT/<n>_gene_related_disease/<disease_id>/<disease_id>_pocket_candidates.csv
DISEASE_DATA_ROOT = PROJECT_ROOT / "separate_disease"
FALLBACK_DISEASE_DATA_ROOTS = [
    PROJECT_ROOT / "separete_disease_no_missing",
]

# Leave as None to evaluate everything found, or set one/both to limit the run.
FOLDER_NAME = None  # example: "34_gene_related_disease"
DISEASE_ID = None   # example: "EFO_0004531"

OUTPUT_ROOT = PROJECT_ROOT / "exactlly_accuracy"
MODULE_OUTPUT_ROOT = OUTPUT_ROOT / MODULE_LABEL

PER_REPEAT_FILENAME = "final_gene_set_exact_accuracy_per_repeat.csv"
SUMMARY_FILENAME = "final_gene_set_exact_accuracy_summary.csv"
OVERALL_FILENAME = "final_gene_set_exact_accuracy_overall.csv"

print(f"Module: {MODULE_LABEL}")
print(f"Iteration roots: {[p.as_posix() for p in ITERATION_OUTPUT_ROOTS]}")
print(f"Disease data root: {DISEASE_DATA_ROOT}")
print(f"Output root: {MODULE_OUTPUT_ROOT}")

Module: 4
Iteration roots: ['/Users/miasmacbook/Desktop/kcl/6-months_project/pocket_model_iterations_4_in_total']
Disease data root: /Users/miasmacbook/Desktop/kcl/6-months_project/separate_disease
Output root: /Users/miasmacbook/Desktop/kcl/6-months_project/exactlly_accuracy/4


In [14]:
from __future__ import annotations

import re
from collections import defaultdict

import pandas as pd

ENSEMBL_RE = re.compile(r"^ENSG\d+(?:\.\d+)?$")
GENE_COUNT_RE = re.compile(r"(\d+)_gene_related_disease")


def normalize_gene_id(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip().split(".")[0]


def read_causal_genes(causal_csv: Path) -> list[str]:
    causal_df = pd.read_csv(causal_csv)
    for column in ["targetId", "target_id", "gene_id", "matrix_gene_id"]:
        if column in causal_df.columns:
            genes = causal_df[column].map(normalize_gene_id)
            return sorted(set(genes.dropna()) - {""})
    raise ValueError(f"Could not find a causal-gene column in {causal_csv}. Columns: {list(causal_df.columns)}")


def parse_final_set(final_set_text) -> list[tuple[str, str]]:
    pairs = []
    if pd.isna(final_set_text):
        return pairs
    for part in str(final_set_text).split(" | "):
        part = part.strip()
        if not part or "->" not in part:
            continue
        seed_gene, selected_label = part.split("->", 1)
        pairs.append((normalize_gene_id(seed_gene), selected_label.strip()))
    return pairs


def build_candidate_lookup(pocket_candidates_csv: Path) -> dict[tuple[str, str], set[str]]:
    candidates = pd.read_csv(pocket_candidates_csv)
    required = {"seed_targetId", "matrix_gene_id"}
    missing = required - set(candidates.columns)
    if missing:
        raise ValueError(f"Missing columns {missing} in {pocket_candidates_csv}")

    lookup = defaultdict(set)
    label_columns = [column for column in ["candidate_label", "matrix_gene_id", "neighbor_symbol"] if column in candidates.columns]
    for _, row in candidates.iterrows():
        seed_gene = normalize_gene_id(row["seed_targetId"])
        matrix_gene = normalize_gene_id(row["matrix_gene_id"])
        if not seed_gene or not matrix_gene:
            continue
        for column in label_columns:
            label = "" if pd.isna(row[column]) else str(row[column]).strip()
            if label:
                lookup[(seed_gene, label)].add(matrix_gene)
                lookup[(seed_gene, normalize_gene_id(label))].add(matrix_gene)
    return lookup


def resolve_final_genes(final_set_text, candidate_lookup: dict[tuple[str, str], set[str]]) -> tuple[list[str], list[str]]:
    resolved_genes = []
    unresolved_labels = []
    for seed_gene, selected_label in parse_final_set(final_set_text):
        matches = set()
        for label in [selected_label, normalize_gene_id(selected_label)]:
            matches.update(candidate_lookup.get((seed_gene, label), set()))
        if not matches and ENSEMBL_RE.match(normalize_gene_id(selected_label)):
            matches.add(normalize_gene_id(selected_label))
        if matches:
            resolved_genes.extend(sorted(matches))
        else:
            unresolved_labels.append(f"{seed_gene}->{selected_label}")
    return sorted(set(resolved_genes)), unresolved_labels


def find_iteration_result_csvs() -> list[Path]:
    paths = []
    for root in ITERATION_OUTPUT_ROOTS:
        if not root.exists():
            print(f"Missing iteration root, skipped: {root}")
            continue
        for path in root.glob("*/*/*_100_change_iterations.csv"):
            if path.name.endswith("_summary.csv"):
                continue
            folder_name = path.parent.parent.name
            disease_id = path.parent.name
            if FOLDER_NAME and folder_name != FOLDER_NAME:
                continue
            if DISEASE_ID and disease_id != DISEASE_ID:
                continue
            paths.append(path)
    return sorted(paths)


def find_final_set_column(df: pd.DataFrame) -> str:
    for column in ["final_set_of_gene", "final set of gene"]:
        if column in df.columns:
            return column
    raise ValueError(f"Could not find final gene-set column. Columns: {list(df.columns)}")


def find_score_column(df: pd.DataFrame) -> str | None:
    for column in ["final_score", "final score"]:
        if column in df.columns:
            return column
    return None


def find_disease_files(folder_name: str, disease_id: str) -> tuple[Path | None, Path | None, Path | None]:
    for root in [DISEASE_DATA_ROOT, *FALLBACK_DISEASE_DATA_ROOTS]:
        disease_dir = root / folder_name / disease_id
        causal_csv = disease_dir / f"{disease_id}.csv"
        pocket_candidates_csv = disease_dir / f"{disease_id}_pocket_candidates.csv"
        if causal_csv.exists() and pocket_candidates_csv.exists():
            return disease_dir, causal_csv, pocket_candidates_csv
    return None, None, None


def gene_count_from_folder(folder_name: str) -> int | None:
    match = GENE_COUNT_RE.match(folder_name)
    return int(match.group(1)) if match else None

In [15]:
per_repeat_rows = []
summary_rows = []
skipped_rows = []

result_csvs = find_iteration_result_csvs()
print(f"Iteration result CSVs found: {len(result_csvs)}")

for result_csv in result_csvs:
    folder_name = result_csv.parent.parent.name
    disease_id = result_csv.parent.name
    n_genes_folder = gene_count_from_folder(folder_name)

    disease_dir, causal_csv, pocket_candidates_csv = find_disease_files(folder_name, disease_id)
    if causal_csv is None or pocket_candidates_csv is None:
        skipped_rows.append({
            "folder_name": folder_name,
            "disease_id": disease_id,
            "source_result_csv": result_csv.as_posix(),
            "reason": "missing causal CSV or pocket candidates CSV",
        })
        continue

    causal_genes = read_causal_genes(causal_csv)
    causal_gene_set = set(causal_genes)
    candidate_lookup = build_candidate_lookup(pocket_candidates_csv)
    results_df = pd.read_csv(result_csv)
    final_set_col = find_final_set_column(results_df)
    score_col = find_score_column(results_df)

    disease_repeat_rows = []
    all_final_genes = set()
    all_unresolved_labels = set()

    for row_index, row in results_df.iterrows():
        final_genes, unresolved_labels = resolve_final_genes(row[final_set_col], candidate_lookup)
        final_gene_set = set(final_genes)
        recovered_causal_genes = sorted(final_gene_set & causal_gene_set)
        all_final_genes.update(final_gene_set)
        all_unresolved_labels.update(unresolved_labels)

        repeat_row = {
            "module_label": MODULE_LABEL,
            "folder_name": folder_name,
            "n_genes_folder": n_genes_folder,
            "disease_id": disease_id,
            "source_result_csv": result_csv.as_posix(),
            "disease_data_dir": disease_dir.as_posix(),
            "seed": row.get("seed", row_index),
            "final_score": row[score_col] if score_col else None,
            "n_causal_genes": len(causal_genes),
            "n_final_genes_resolved": len(final_genes),
            "n_causal_genes_in_final_set": len(recovered_causal_genes),
            "any_causal_gene_in_final_set": len(recovered_causal_genes) > 0,
            "causal_gene_recovery_percent": (len(recovered_causal_genes) / len(causal_genes) * 100) if causal_genes else 0,
            "causal_genes_in_final_set": "|".join(recovered_causal_genes),
            "final_genes_resolved": "|".join(final_genes),
            "unresolved_final_set_labels": "|".join(unresolved_labels),
        }
        disease_repeat_rows.append(repeat_row)
        per_repeat_rows.append(repeat_row)

    found_causal_after_100 = sorted(all_final_genes & causal_gene_set)
    missing_causal_after_100 = sorted(causal_gene_set - all_final_genes)
    whole_causal_gene_set_found = len(missing_causal_after_100) == 0

    disease_repeat_df = pd.DataFrame(disease_repeat_rows)
    summary_rows.append({
        "module_label": MODULE_LABEL,
        "folder_name": folder_name,
        "n_genes_folder": n_genes_folder,
        "disease_id": disease_id,
        "source_result_csv": result_csv.as_posix(),
        "disease_data_dir": disease_dir.as_posix(),
        "n_repeats": len(disease_repeat_df),
        "n_causal_genes": len(causal_genes),
        "n_unique_final_genes_after_100_repeats": len(all_final_genes),
        "n_causal_genes_found_after_100_repeats": len(found_causal_after_100),
        "n_causal_genes_missing_after_100_repeats": len(missing_causal_after_100),
        "any_causal_gene_found_after_100_repeats": len(found_causal_after_100) > 0,
        "whole_causal_gene_set_found_after_100_repeats": whole_causal_gene_set_found,
        "whole_causal_gene_set_accuracy_percent": 100.0 if whole_causal_gene_set_found else 0.0,
        "causal_gene_set_recovery_percent_after_100_repeats": (len(found_causal_after_100) / len(causal_genes) * 100) if causal_genes else 0,
        "repeat_count_with_any_causal_gene": int(disease_repeat_df["any_causal_gene_in_final_set"].sum()),
        "accuracy_any_causal_gene_percent": float(disease_repeat_df["any_causal_gene_in_final_set"].mean() * 100) if len(disease_repeat_df) else 0,
        "mean_repeat_causal_gene_recovery_percent": float(disease_repeat_df["causal_gene_recovery_percent"].mean()) if len(disease_repeat_df) else 0,
        "causal_genes": "|".join(causal_genes),
        "causal_genes_found_after_100_repeats": "|".join(found_causal_after_100),
        "causal_genes_missing_after_100_repeats": "|".join(missing_causal_after_100),
        "all_final_genes_after_100_repeats": "|".join(sorted(all_final_genes)),
        "unresolved_final_set_labels_after_100_repeats": "|".join(sorted(all_unresolved_labels)),
    })

per_repeat_df = pd.DataFrame(per_repeat_rows)
summary_df = pd.DataFrame(summary_rows)
skipped_df = pd.DataFrame(skipped_rows)

print(f"Evaluated diseases: {summary_df['disease_id'].nunique() if not summary_df.empty else 0}")
print(f"Evaluated repeat rows: {len(per_repeat_df)}")
print(f"Skipped result CSVs: {len(skipped_df)}")
summary_df.head()

Iteration result CSVs found: 602
Evaluated diseases: 602
Evaluated repeat rows: 60200
Skipped result CSVs: 0


,module_label,folder_name,n_genes_folder,disease_id,source_result_csv,disease_data_dir,n_repeats,n_causal_genes,n_unique_final_genes_after_100_repeats,n_causal_genes_found_after_100_repeats,...,whole_causal_gene_set_accuracy_percent,causal_gene_set_recovery_percent_after_100_repeats,repeat_count_with_any_causal_gene,accuracy_any_causal_gene_percent,mean_repeat_causal_gene_recovery_percent,causal_genes,causal_genes_found_after_100_repeats,causal_genes_missing_after_100_repeats,all_final_genes_after_100_repeats,unresolved_final_set_labels_after_100_repeats
0,4,102_gene_related_disease,102,EFO_0011011,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,102,494,102,...,100.0,100.0,100,100.0,20.490196,ENSG00000025293|ENSG00000049192|ENSG0000004954...,ENSG00000025293|ENSG00000049192|ENSG0000004954...,,ENSG00000007312|ENSG00000007314|ENSG0000000822...,
1,4,10_gene_related_disease,10,EFO_0000095,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,43,10,...,100.0,100.0,99,99.0,40.100000,ENSG00000082898|ENSG00000115524|ENSG0000014151...,ENSG00000082898|ENSG00000115524|ENSG0000014151...,,ENSG00000008226|ENSG00000060971|ENSG0000008289...,
2,4,10_gene_related_disease,10,EFO_0000275,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,41,7,...,0.0,70.0,97,97.0,28.800000,ENSG00000095637|ENSG00000103024|ENSG0000011713...,ENSG00000095637|ENSG00000103024|ENSG0000011713...,ENSG00000140986|ENSG00000155657|ENSG00000166317,ENSG00000059573|ENSG00000066336|ENSG0000007407...,
3,4,10_gene_related_disease,10,EFO_0001072,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,11,43,11,...,100.0,100.0,100,100.0,38.545455,ENSG00000105392|ENSG00000113494|ENSG0000011381...,ENSG00000105392|ENSG00000113494|ENSG0000011381...,,ENSG00000083812|ENSG00000105392|ENSG0000011349...,
4,4,10_gene_related_disease,10,EFO_0002422,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,37,9,...,0.0,90.0,99,99.0,44.100000,ENSG00000095002|ENSG00000099949|ENSG0000011606...,ENSG00000095002|ENSG00000099949|ENSG0000011606...,ENSG00000134982,ENSG00000095002|ENSG00000099949|ENSG0000010020...,


In [16]:
written_outputs = []
MODULE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if per_repeat_df.empty or summary_df.empty:
    print("No rows were evaluated. Check ITERATION_OUTPUT_ROOTS and disease-data paths.")
else:
    for folder_name, folder_repeat_df in per_repeat_df.groupby("folder_name", sort=True):
        folder_output_dir = MODULE_OUTPUT_ROOT / folder_name
        folder_output_dir.mkdir(parents=True, exist_ok=True)

        folder_summary_df = summary_df[summary_df["folder_name"] == folder_name].sort_values("disease_id")
        per_repeat_path = folder_output_dir / PER_REPEAT_FILENAME
        summary_path = folder_output_dir / SUMMARY_FILENAME

        folder_repeat_df.sort_values(["disease_id", "seed"]).to_csv(per_repeat_path, index=False)
        folder_summary_df.to_csv(summary_path, index=False)
        written_outputs.extend([per_repeat_path, summary_path])

    all_per_repeat_path = MODULE_OUTPUT_ROOT / f"all_{PER_REPEAT_FILENAME}"
    all_summary_path = MODULE_OUTPUT_ROOT / f"all_{SUMMARY_FILENAME}"
    per_repeat_df.sort_values(["n_genes_folder", "disease_id", "seed"]).to_csv(all_per_repeat_path, index=False)
    summary_df.sort_values(["n_genes_folder", "disease_id"]).to_csv(all_summary_path, index=False)
    written_outputs.extend([all_per_repeat_path, all_summary_path])

    overall_df = pd.DataFrame([{
        "module_label": MODULE_LABEL,
        "n_diseases": summary_df["disease_id"].nunique(),
        "n_repeat_rows": len(per_repeat_df),
        "disease_count_with_any_causal_gene_after_100_repeats": int(summary_df["any_causal_gene_found_after_100_repeats"].sum()),
        "accuracy_any_causal_gene_after_100_repeats_percent": float(summary_df["any_causal_gene_found_after_100_repeats"].mean() * 100),
        "disease_count_with_whole_causal_gene_set_after_100_repeats": int(summary_df["whole_causal_gene_set_found_after_100_repeats"].sum()),
        "accuracy_whole_causal_gene_set_after_100_repeats_percent": float(summary_df["whole_causal_gene_set_found_after_100_repeats"].mean() * 100),
        "mean_causal_gene_set_recovery_percent_after_100_repeats": float(summary_df["causal_gene_set_recovery_percent_after_100_repeats"].mean()),
        "mean_repeat_accuracy_any_causal_gene_percent": float(summary_df["accuracy_any_causal_gene_percent"].mean()),
    }])
    overall_path = MODULE_OUTPUT_ROOT / OVERALL_FILENAME
    overall_df.to_csv(overall_path, index=False)
    written_outputs.append(overall_path)

    if not skipped_df.empty:
        skipped_path = MODULE_OUTPUT_ROOT / "skipped_iteration_result_csvs.csv"
        skipped_df.to_csv(skipped_path, index=False)
        written_outputs.append(skipped_path)

    print(f"Saved {len(written_outputs)} output files under: {MODULE_OUTPUT_ROOT}")
    print(f"All repeats CSV: {all_per_repeat_path}")
    print(f"All summary CSV: {all_summary_path}")
    print(f"Overall CSV: {overall_path}")

    display(overall_df)
    display(summary_df.sort_values(["whole_causal_gene_set_found_after_100_repeats", "causal_gene_set_recovery_percent_after_100_repeats"], ascending=False).head(20))

Saved 117 output files under: /Users/miasmacbook/Desktop/kcl/6-months_project/exactlly_accuracy/4
All repeats CSV: /Users/miasmacbook/Desktop/kcl/6-months_project/exactlly_accuracy/4/all_final_gene_set_exact_accuracy_per_repeat.csv
All summary CSV: /Users/miasmacbook/Desktop/kcl/6-months_project/exactlly_accuracy/4/all_final_gene_set_exact_accuracy_summary.csv
Overall CSV: /Users/miasmacbook/Desktop/kcl/6-months_project/exactlly_accuracy/4/final_gene_set_exact_accuracy_overall.csv


,module_label,n_diseases,n_repeat_rows,disease_count_with_any_causal_gene_after_100_repeats,accuracy_any_causal_gene_after_100_repeats_percent,disease_count_with_whole_causal_gene_set_after_100_repeats,accuracy_whole_causal_gene_set_after_100_repeats_percent,mean_causal_gene_set_recovery_percent_after_100_repeats,mean_repeat_accuracy_any_causal_gene_percent
0,4,602,60200,590,98.006645,286,47.508306,83.012704,73.797342


,module_label,folder_name,n_genes_folder,disease_id,source_result_csv,disease_data_dir,n_repeats,n_causal_genes,n_unique_final_genes_after_100_repeats,n_causal_genes_found_after_100_repeats,...,whole_causal_gene_set_accuracy_percent,causal_gene_set_recovery_percent_after_100_repeats,repeat_count_with_any_causal_gene,accuracy_any_causal_gene_percent,mean_repeat_causal_gene_recovery_percent,causal_genes,causal_genes_found_after_100_repeats,causal_genes_missing_after_100_repeats,all_final_genes_after_100_repeats,unresolved_final_set_labels_after_100_repeats
0,4,102_gene_related_disease,102,EFO_0011011,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,102,494,102,...,100.0,100.0,100,100.0,20.490196,ENSG00000025293|ENSG00000049192|ENSG0000004954...,ENSG00000025293|ENSG00000049192|ENSG0000004954...,,ENSG00000007312|ENSG00000007314|ENSG0000000822...,
1,4,10_gene_related_disease,10,EFO_0000095,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,43,10,...,100.0,100.0,99,99.0,40.100000,ENSG00000082898|ENSG00000115524|ENSG0000014151...,ENSG00000082898|ENSG00000115524|ENSG0000014151...,,ENSG00000008226|ENSG00000060971|ENSG0000008289...,
3,4,10_gene_related_disease,10,EFO_0001072,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,11,43,11,...,100.0,100.0,100,100.0,38.545455,ENSG00000105392|ENSG00000113494|ENSG0000011381...,ENSG00000105392|ENSG00000113494|ENSG0000011381...,,ENSG00000083812|ENSG00000105392|ENSG0000011349...,
6,4,10_gene_related_disease,10,EFO_0004198,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,11,49,11,...,100.0,100.0,100,100.0,30.727273,ENSG00000008710|ENSG00000083799|ENSG0000010404...,ENSG00000008710|ENSG00000083799|ENSG0000010404...,,ENSG00000008710|ENSG00000065057|ENSG0000007391...,
7,4,10_gene_related_disease,10,EFO_0004614,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,45,10,...,100.0,100.0,74,74.0,27.600000,ENSG00000073060|ENSG00000087237|ENSG0000010167...,ENSG00000073060|ENSG00000087237|ENSG0000010167...,,ENSG00000051108|ENSG00000070214|ENSG0000007306...,
8,4,10_gene_related_disease,10,EFO_0006807,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,49,10,...,100.0,100.0,74,74.0,36.600000,ENSG00000015520|ENSG00000084674|ENSG0000010167...,ENSG00000015520|ENSG00000084674|ENSG0000010167...,,ENSG00000015520|ENSG00000070214|ENSG0000007955...,
9,4,10_gene_related_disease,10,EFO_0006925,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,30,10,...,100.0,100.0,100,100.0,24.000000,ENSG00000026652|ENSG00000112110|ENSG0000011249...,ENSG00000026652|ENSG00000112110|ENSG0000011249...,,ENSG00000005059|ENSG00000026652|ENSG0000008551...,
10,4,10_gene_related_disease,10,EFO_0007009,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,37,10,...,100.0,100.0,100,100.0,41.500000,ENSG00000077498|ENSG00000104044|ENSG0000010716...,ENSG00000077498|ENSG00000104044|ENSG0000010716...,,ENSG00000077498|ENSG00000082196|ENSG0000008699...,
11,4,10_gene_related_disease,10,EFO_0008595,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,49,10,...,100.0,100.0,34,34.0,7.800000,ENSG00000015520|ENSG00000084674|ENSG0000011024...,ENSG00000015520|ENSG00000084674|ENSG0000011024...,,ENSG00000015520|ENSG00000062370|ENSG0000007955...,
16,4,10_gene_related_disease,10,EFO_0021902,/Users/miasmacbook/Desktop/kcl/6-months_projec...,/Users/miasmacbook/Desktop/kcl/6-months_projec...,100,10,47,10,...,100.0,100.0,53,53.0,11.600000,ENSG00000060566|ENSG00000084674|ENSG0000011024...,ENSG00000060566|ENSG00000084674|ENSG0000011024...,,ENSG00000060566|ENSG00000077463|ENSG0000008467...,


In [17]:
# Optional: grouped view by disease gene count, similar to accuracy_by_gene_count.ipynb.
if summary_df.empty:
    print("No summary rows available.")
else:
    grouped_df = (
        summary_df.groupby("n_genes_folder", as_index=False)
        .agg(
            n_diseases=("disease_id", "nunique"),
            mean_whole_causal_gene_set_accuracy_percent=("whole_causal_gene_set_accuracy_percent", "mean"),
            mean_causal_gene_set_recovery_percent_after_100_repeats=("causal_gene_set_recovery_percent_after_100_repeats", "mean"),
            mean_accuracy_any_causal_gene_percent=("accuracy_any_causal_gene_percent", "mean"),
            min_causal_gene_set_recovery_percent_after_100_repeats=("causal_gene_set_recovery_percent_after_100_repeats", "min"),
            max_causal_gene_set_recovery_percent_after_100_repeats=("causal_gene_set_recovery_percent_after_100_repeats", "max"),
        )
        .sort_values("n_genes_folder")
        .reset_index(drop=True)
    )

    grouped_path = MODULE_OUTPUT_ROOT / "final_gene_set_exact_accuracy_by_gene_count_grouped.csv"
    grouped_df.to_csv(grouped_path, index=False)
    print(f"Saved grouped CSV: {grouped_path}")

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    output_png = MODULE_OUTPUT_ROOT / "final_gene_set_exact_accuracy_by_gene_count.png"

    fig, ax = plt.subplots(figsize=(11, 6.5))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    ax.scatter(
        summary_df["n_genes_folder"],
        summary_df["whole_causal_gene_set_accuracy_percent"],
        alpha=0.35,
        color="#2563eb",
        label="Each disease",
    )

    ax.plot(
        grouped_df["n_genes_folder"],
        grouped_df["mean_whole_causal_gene_set_accuracy_percent"],
        marker="o",
        linewidth=2.5,
        color="#dc2626",
        label="Mean whole-set accuracy",
    )

    ax.set_xlabel("Number of genes related to disease", fontsize=12)
    ax.set_ylabel("Accuracy: 100-repeat final sets recover whole causal gene set (%)", fontsize=12)
    ax.set_title("Exact final-set accuracy by disease gene count", fontsize=15, pad=12)
    ax.set_ylim(0, 105)
    ax.grid(True, axis="y", color="#cbd5e1", linewidth=0.8, alpha=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, loc="best")

    plt.tight_layout()
    plt.savefig(output_png, dpi=150)
    plt.close(fig)

    print(f"Saved graph: {output_png}")
    display(grouped_df)

Saved grouped CSV: /Users/miasmacbook/Desktop/kcl/6-months_project/exactlly_accuracy/4/final_gene_set_exact_accuracy_by_gene_count_grouped.csv
Saved graph: /Users/miasmacbook/Desktop/kcl/6-months_project/exactlly_accuracy/4/final_gene_set_exact_accuracy_by_gene_count.png


,n_genes_folder,n_diseases,mean_whole_causal_gene_set_accuracy_percent,mean_causal_gene_set_recovery_percent_after_100_repeats,mean_accuracy_any_causal_gene_percent,min_causal_gene_set_recovery_percent_after_100_repeats,max_causal_gene_set_recovery_percent_after_100_repeats
0,3,113,32.743363,60.766962,58.141593,0.000000,100.000000
1,4,72,34.722222,72.916667,78.152778,0.000000,100.000000
2,5,39,28.205128,78.632479,79.153846,20.000000,100.000000
3,6,41,43.902439,81.300813,76.536585,33.333333,100.000000
4,7,37,43.243243,85.810811,70.621622,28.571429,100.000000
5,8,41,68.292683,90.955285,64.048780,37.500000,100.000000
6,9,66,63.636364,93.282828,68.984848,55.555556,100.000000
7,10,47,68.085106,93.191489,69.489362,50.000000,100.000000
8,11,14,57.142857,95.454545,70.000000,81.818182,100.000000
9,12,18,77.777778,96.296296,69.111111,75.000000,100.000000
